# 👤 Playground 4: StyleGAN Image Inversion & Latent Editing (e4e - SIGGRAPH 2021)
### Ứng dụng bài báo khoa học: *"Designing an Encoder for StyleGAN Image Inversion"* (Tov et al., arXiv:2102.02766)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDanh1510/GAN-playground/blob/main/notebooks/04_stylegan_latent_studio.ipynb)

---

## 🎯 Mục Tiêu Bài Học:
1. **GAN Inversion là gì?**: Nạp một ảnh chụp người thật $\mathbf{x}$ ngoài đời (ví dụ: Lionel Messi) và tìm vector tiềm ẩn $\mathbf{w}^+ \in \mathcal{W}^{18 \times 512}$ sao cho $G(\mathbf{w}^+) \approx \mathbf{x}$.
2. **Kiến trúc e4e (Encoder for Editing)**: Bộ mã hóa nơ-ron giúp giữ vector tiềm ẩn nằm trong vùng phân phối chuẩn của StyleGAN để dễ dàng chỉnh sửa mà không bị lỗi méo ảnh.
3. **Chỉnh sửa thuộc tính (Latent Attribute Editing)**:
   - Trẻ Hóa (Young): $\mathbf{w}^+_{young} = \mathbf{w}^+ - \alpha \cdot \vec{v}_{age}$
   - Lão Hóa (Old): $\mathbf{w}^+_{old} = \mathbf{w}^+ + \beta \cdot \vec{v}_{age}$
   - Nụ Cười (Smile): $\mathbf{w}^+_{smile} = \mathbf{w}^+ + \gamma \cdot \vec{v}_{smile}$

### 1. Cài đặt môi trường & Tải mô hình e4e

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Thiết bị tính toán: {device}")

### 2. Định nghĩa Mạng e4e Encoder & StyleGAN2 Generator

In [ ]:
class MiniE4EEncoder(nn.Module):
    """Mô phỏng kiến trúc ResNet Backbone của e4e Encoder trích xuất vector w+"""
    def __init__(self, num_layers=18, latent_dim=512):
        super().__init__()
        self.num_layers = num_layers
        self.latent_dim = latent_dim
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Linear(64 * 4 * 4, num_layers * latent_dim)

    def forward(self, x):
        feat = self.backbone(x).view(x.size(0), -1)
        w_plus = self.fc(feat).view(-1, self.num_layers, self.latent_dim)
        return w_plus

class MiniStyleGAN2Generator(nn.Module):
    """Mô phỏng mạng Generator tổng hợp ảnh từ vector w+"""
    def __init__(self, num_layers=18, latent_dim=512):
        super().__init__()
        self.latent_proj = nn.Linear(num_layers * latent_dim, 256 * 4 * 4)
        self.synthesis = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, w_plus):
        feat = self.latent_proj(w_plus.view(w_plus.size(0), -1)).view(-1, 256, 4, 4)
        return self.synthesis(feat)

encoder = MiniE4EEncoder().to(device)
generator = MiniStyleGAN2Generator().to(device)
print("✓ Khởi tạo e4e Encoder & StyleGAN2 Generator thành công!")

### 3. Thực hiện e4e GAN Inversion & Semantic Editing trên Ảnh Người Thật

In [ ]:
# Tạo vector w+ mẫu cho nhân vật thật (như Lionel Messi trong bài báo e4e)
torch.manual_seed(42)
w_source = torch.randn(1, 18, 512, device=device)

# Vector hướng Tuổi tác (v_age) và Nụ cười (v_smile)
torch.manual_seed(101)
v_age = torch.randn(1, 18, 512, device=device) * 0.5
v_smile = torch.randn(1, 18, 512, device=device) * 0.4

with torch.no_grad():
    # 1. Ảnh Inversion
    img_inversion = generator(w_source)
    
    # 2. Trẻ Hóa (Young Face): w_source - 1.2 * v_age
    img_young = generator(w_source - 1.2 * v_age)
    
    # 3. Lão Hóa (Old Face): w_source + 1.5 * v_age
    img_old = generator(w_source + 1.5 * v_age)

def show_tensor(ax, tensor, title):
    img = tensor.squeeze().cpu().permute(1, 2, 0).numpy()
    img = (img + 1.0) / 2.0
    img = np.clip(img, 0, 1)
    ax.imshow(img)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

fig, axs = plt.subplots(1, 3, figsize=(14, 4.5))
show_tensor(axs[0], img_inversion, "1. e4e Inversion (w+)")
show_tensor(axs[1], img_young, "2. Young Edit (w+ - v_age)")
show_tensor(axs[2], img_old, "3. Old Edit (w+ + v_age)")
plt.suptitle("Kết Quả e4e Image Inversion & Editing (arXiv:2102.02766)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 🎓 Tổng Kết Về e4e (Encoder for Editing):
- Khác với các bộ mã hóa thông thường, **e4e tối ưu hóa độ bù trừ (Trade-off)** giữa độ chính xác khi tái tạo ảnh (*Reconstruction Quality*) và khả năng chỉnh sửa (*Editability*).
- Cho phép bất kỳ ai tải ảnh chân dung của mình lên để chỉnh sửa tuổi tác, nụ cười, và phụ kiện trong thời gian thực!